In [ ]:
# Install libraries (restart session afterwards)
!pip install facenet-pytorch opencv-python-headless torchvision > /dev/null 2>&1
!pip install deepface > /dev/null 2>&1
!pip install mediapipe > /dev/null 2>&1
!pip install tensorflow-cpu > /dev/null 2>&1
!pip install transformers > /dev/null 2>&1
!pip install torch==2.7.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 > /dev/null 2>&1
!pip install transformers==4.51.3 > /dev/null 2>&1


In [ ]:
# import libraries
import os
import re
from tqdm import tqdm
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import tensorflow as tf
from sklearn.metrics.pairwise import cosine_similarity
from facenet_pytorch import MTCNN, InceptionResnetV1
from deepface import DeepFace
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    AutoProcessor,
    CLIPModel,
    AutoModel
)
import mediapipe as mp
from google.colab import drive

In [ ]:
# Mount Drive
drive.mount('/content/drive')

## Get images

In [ ]:
import pandas as pd

#strip paths from /content/drive/MyDrive/Thesis/images_generated/bbc_images_0006_139_2.jpg' to ./bbc/images/0006/139.jpg
def strip_path(path):
    filename = path.split('/')[-1]
    parts = filename.replace('.jpg', '').split('_')
    prefix = '_'.join(parts[:-4])
    return f"./{prefix}/images/{parts[-3]}/{parts[-2]}.jpg"

df_generated_images = pd.read_excel('/content/drive/MyDrive/Thesis/generated_images.xlsx')
df_generated_images['image_path_stripped'] = df_generated_images['image_path'].apply(strip_path)

grouped_counts = df_generated_images.groupby('image_path_stripped').size().reset_index(name='count')

# get images that have 5 generated images
fullgeneratedimages = grouped_counts[grouped_counts['count'] == 5]
fullgeneratedimages


In [ ]:
def get_images(image_paths):
    if isinstance(image_paths, str):
        image_paths = [image_paths]
    all_images_paths = []

    for image_path in image_paths:
        images_paths = []
        image_path_strip = image_path.strip().lstrip('./')
        # get original path
        original_image_path = f'/content/drive/MyDrive/Thesis/images_dataset/origin/{image_path_strip}'

        # set standard for generated path
        parts = image_path_strip.split('/')
        image_name_path_based = f"{parts[0]}_{parts[1]}_{parts[2]}_{parts[3].split('.')[0]}_X.jpg"

        # create base for generated path
        base_generated_path = '/content/drive/MyDrive/Thesis/images_generated/'

        images_paths.append(original_image_path)

        # get all genenerated paths based on standard path
        for i in range(1, 6):
            gen_image_name = image_name_path_based.replace('X', str(i))
            gen_image_path = base_generated_path + gen_image_name
            images_paths.append(gen_image_path)
            print(gen_image_path)

        all_images_paths.extend(images_paths)

    return all_images_paths

# get all paths of generated images
images_paths = get_images(fullgeneratedimages['image_path_stripped'])
images_paths = list(set(images_paths))

## Set functions for Face Recognitions

In [ ]:
def get_actor_names_and_labels():
    df = pd.read_parquet("hf://datasets/tonyassi/celebrity-1000/data/train-00000-of-00001.parquet")

    def clean_name_from_path(path):
        name = path.replace('-', ' ')
        name = re.sub(r'\d+', '', name)
        name = name.replace('.jpg', '')
        return name.strip()

    df['actor_name'] = df['image'].apply(lambda x: clean_name_from_path(x['path']))
    df_unique = df.drop_duplicates(subset=['actor_name', 'label'])
    return df_unique

# Get embeddings from folder
def get_embeddings_for_folder(folder_path):
    embeddings = []
    for filename in os.listdir(folder_path):
        if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        try:
            path = os.path.join(folder_path, filename)
            img = Image.open(path).convert('RGB')
            faces = mtcnn(img)
            if faces is None or len(faces) == 0:
                print(f"No face detected in {filename}")
                continue

            for i in range(faces.shape[0]):
                face = faces[i].unsqueeze(0).to(device)
                with torch.no_grad():
                    embedding = resnet(face)

                embeddings.append(embedding.squeeze(0).cpu().numpy())
        except Exception as e:
            print(f"Failed on {filename}: {e}")

    if not embeddings:
        return np.zeros((0, 512))
    return np.vstack(embeddings)


# Match embeddings
def is_match(embedding, known_embeddings, threshold=0.90):
    sims = [cosine_similarity([embedding], [known_emb])[0][0] for known_emb in known_embeddings]
    max_sim = max(sims)
    return (max_sim > threshold, max_sim)

# Process pose landmarks
def process_observation_for_actor(image, actor_name):
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)
    selected_keypoints = [
        mp_pose.PoseLandmark.NOSE,
        mp_pose.PoseLandmark.LEFT_EYE,
        mp_pose.PoseLandmark.RIGHT_EYE,
        mp_pose.PoseLandmark.LEFT_EAR,
        mp_pose.PoseLandmark.RIGHT_EAR,
        mp_pose.PoseLandmark.LEFT_SHOULDER,
        mp_pose.PoseLandmark.RIGHT_SHOULDER,
        mp_pose.PoseLandmark.LEFT_ELBOW,
        mp_pose.PoseLandmark.RIGHT_ELBOW,
        mp_pose.PoseLandmark.LEFT_WRIST,
        mp_pose.PoseLandmark.RIGHT_WRIST
    ]

    if not results.pose_landmarks:
        return None

    keypoints = []
    for joint in selected_keypoints:
        landmark = results.pose_landmarks.landmark[joint]
        if landmark.visibility > 0.1:
            keypoints.append([round(landmark.x, 2), round(landmark.y, 2)])

        else:
            keypoints.append(None)

    names = [
        "NOSE", "LEFT_EYE", "RIGHT_EYE", "LEFT_EAR", "RIGHT_EAR",
        "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_ELBOW", "RIGHT_ELBOW",
        "LEFT_WRIST", "RIGHT_WRIST"
    ]
    return {f"{name}_{actor_name}": point for name, point in zip(names, keypoints)}

In [ ]:
#set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load celebrity labels from tonyassi
df_celebrity_names = get_actor_names_and_labels()

# Setup pose model
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Initialize models
mtcnn = MTCNN(keep_all=True)
processor = AutoImageProcessor.from_pretrained("tonyassi/celebrity-classifier")
model = AutoModelForImageClassification.from_pretrained("tonyassi/celebrity-classifier")

resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)
resnet.fc = torch.nn.Identity()
resnet.eval()
device = torch.device("cpu")
resnet.to(device)

# Load Merkel and Rutte embeddings (precomputed arrays)
embedding_dir = '/content/drive/MyDrive/Thesis/images_politicians_embeddings'
rutte_embeddings = get_embeddings_for_folder(os.path.join(embedding_dir, 'rutte'))
merkel_embeddings = get_embeddings_for_folder(os.path.join(embedding_dir, 'merkel'))

# Label mapping for tonyassi
name_to_label = {
    "donald trump": df_celebrity_names[df_celebrity_names['actor_name'] == "Donald Trump"].iloc[0]['label'],
    "barack obama": df_celebrity_names[df_celebrity_names['actor_name'] == "Barack Obama"].iloc[0]['label']
}

## Detect Face centers

In [ ]:
def detect_politician_face_features(image_paths):
    data = []
    politicians = ['donald trump', 'barack obama', 'angela merkel', 'mark rutte']

    for image_path in image_paths:
        image = cv2.imread(image_path)
        if image is None:
            print(f"Could not load image: {image_path}")
            continue
        print(image_path)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w, _ = image_rgb.shape
        boxes, _ = mtcnn.detect(image_rgb)
        face_boxes = []

        # Run pose detection on full image once
        pose_result = pose.process(image_rgb)
        pose_landmarks = pose_result.pose_landmarks.landmark if pose_result.pose_landmarks else None

        # Track recognized politicians
        pol_data = {
            name: {
                "box": None,
                "center": None,
                "emotion": None,
                "emotion_conf": None,
                "landmarks": {}
            } for name in politicians
        }

        if boxes is not None:
            faces = mtcnn(Image.fromarray(image_rgb))
            for i, face_tensor in enumerate(faces):
                if face_tensor is None:
                    continue
                x1, y1, x2, y2 = [int(coord) for coord in boxes[i]]
                cx = (x1 + x2) / 2
                cy = (y1 + y2) / 2
                cx_factor = round(cx / w, 4)
                cy_factor = round(cy / h, 4)
                box_factor = [
                    round(x1 / w, 4),
                    round(y1 / h, 4),
                    round(x2 / w, 4),
                    round(y2 / h, 4)
                ]
                face_boxes.append(box_factor)
                cropped_face = image_rgb[y1:y2, x1:x2]

                # Hugging Face classifier (Obama, Trump)
                inputs = processor(images=cropped_face, return_tensors="pt")
                with torch.no_grad():
                    outputs = model(**inputs)
                logits = outputs.logits.detach()
                probs = torch.nn.functional.softmax(logits, dim=-1).numpy()
                pred_idx = logits.argmax(-1).item()
                conf = probs[0, pred_idx]

                matched_name = None
                for name, label in name_to_label.items():
                    if pred_idx == label and conf >= 0.90:
                        matched_name = name
                        break

                # Merkel / Rutte via embeddings
                if not matched_name:
                    embedding = resnet(face_tensor.unsqueeze(0).to(device)).detach().cpu().numpy()[0]
                    if is_match(embedding, rutte_embeddings)[0]:
                        matched_name = 'mark rutte'
                    elif is_match(embedding, merkel_embeddings )[0]:
                        matched_name = 'angela merkel'

                if matched_name:
                    matched_name = matched_name.lower()
                    if matched_name not in pol_data:
                        continue
                    pol_data[matched_name]["box"] = box_factor
                    pol_data[matched_name]["center"] = [cx_factor, cy_factor]

                    # Emotion detection (corrected block)
                    try:
                        if cropped_face.shape[0] == 0 or cropped_face.shape[1] == 0:
                            raise ValueError("Empty face")

                        face_pil = Image.fromarray(cropped_face)
                        result = DeepFace.analyze(np.array(face_pil), actions=["emotion"], enforce_detection=False)

                        if isinstance(result, list):
                            result = result[0]

                        dom = result["dominant_emotion"]
                        conf_em = result["emotion"][dom]
                        pol_data[matched_name]["emotion"] = dom
                        pol_data[matched_name]["emotion_conf"] = round(conf_em, 2)

                    except Exception as e:
                        print(f"Emotion detection failed for {matched_name}: {e}")
                        pol_data[matched_name]["emotion"] = "error"
                        pol_data[matched_name]["emotion_conf"] = 0.0

                    # Match pose landmarks to face
                    if pose_landmarks:
                        nose_lm = pose_landmarks[mp.solutions.pose.PoseLandmark.NOSE]
                        nose_x = nose_lm.x * w
                        nose_y = nose_lm.y * h
                        dist = ((nose_x - cx) ** 2 + (nose_y - cy) ** 2) ** 0.5
                        if dist < 100:
                            for joint in [
                                mp.solutions.pose.PoseLandmark.NOSE,
                                mp.solutions.pose.PoseLandmark.LEFT_EYE,
                                mp.solutions.pose.PoseLandmark.RIGHT_EYE,
                                mp.solutions.pose.PoseLandmark.LEFT_EAR,
                                mp.solutions.pose.PoseLandmark.RIGHT_EAR,
                                mp.solutions.pose.PoseLandmark.LEFT_SHOULDER,
                                mp.solutions.pose.PoseLandmark.RIGHT_SHOULDER,
                                mp.solutions.pose.PoseLandmark.LEFT_ELBOW,
                                mp.solutions.pose.PoseLandmark.RIGHT_ELBOW,
                                mp.solutions.pose.PoseLandmark.LEFT_WRIST,
                                mp.solutions.pose.PoseLandmark.RIGHT_WRIST,
                            ]:
                                lm = pose_landmarks[joint]
                                if lm.visibility > 0.1:
                                    x_factor = round(lm.x, 4)
                                    y_factor = round(lm.y, 4)
                                    short = matched_name.replace(" ", "_")
                                    key = f"{joint.name.lower()}_face_{short}"
                                    pol_data[matched_name]["landmarks"][key] = (x_factor, y_factor)

        row = {
            "image_path": image_path,
            "count_of_faces": len(face_boxes),
            "face_boxes": face_boxes
        }
        for name in politicians:
            short = name.lower().replace(" ", "_")
            row[f"box_face_{short}"] = pol_data[name]["box"]
            row[f"center_face_{short}"] = pol_data[name]["center"]
            row[f"emotion_face_{short}"] = pol_data[name]["emotion"]
            row[f"emotion_conf_face_{short}"] = pol_data[name]["emotion_conf"]
            for key, val in pol_data[name]["landmarks"].items():
                row[key] = val

        data.append(row)

    return pd.DataFrame(data)


In [ ]:
df_results_face = detect_politician_face_features(images_paths)

In [ ]:
df_results_face = df_results_face[['image_path',	'count_of_faces',	'face_boxes',	'box_face_donald_trump',	'center_face_donald_trump',	'emotion_face_donald_trump','emotion_conf_face_donald_trump',
 'box_face_barack_obama',
       'center_face_barack_obama', 'emotion_face_barack_obama',
       'emotion_conf_face_barack_obama', 'box_face_angela_merkel',
       'center_face_angela_merkel', 'emotion_face_angela_merkel',
       'emotion_conf_face_angela_merkel', 'box_face_mark_rutte',
       'center_face_mark_rutte', 'emotion_face_mark_rutte',
       'emotion_conf_face_mark_rutte'

]]

## Detect Body Parts

In [ ]:
def detect_landmarks(images_paths):
    landmarks_data = []
    politicians = ["donald trump", "barack obama", "angela merkel", "mark rutte"]

    for image_path in images_paths:
        image = cv2.imread(image_path)
        if image is None:
            print(f"Unable to read image {image_path}")
            continue

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        boxes, _ = mtcnn.detect(image_rgb)

        image_landmarks = {"image_path": image_path}
        for name in politicians:
            key = name.replace(" ", "_")
            for joint in [
                "NOSE", "LEFT_EYE", "RIGHT_EYE", "LEFT_EAR", "RIGHT_EAR",
                "LEFT_SHOULDER", "RIGHT_SHOULDER", "LEFT_ELBOW", "RIGHT_ELBOW",
                "LEFT_WRIST", "RIGHT_WRIST"
            ]:
                image_landmarks[f"{joint}_{key}"] = None

        if boxes is not None:
            faces = mtcnn(Image.fromarray(image_rgb))
            for i, face_tensor in enumerate(faces):
                if face_tensor is None:
                    continue
                x1, y1, x2, y2 = [int(coord) for coord in boxes[i]]
                cropped_face = image_rgb[y1:y2, x1:x2]

                # Hugging Face classifier
                inputs = processor(images=cropped_face, return_tensors="pt")
                with torch.no_grad():
                    outputs = model(**inputs)
                logits = outputs.logits.detach()
                probs = torch.nn.functional.softmax(logits, dim=-1).numpy()
                pred_idx = logits.argmax(-1).item()
                conf = probs[0, pred_idx]

                matched_name = None
                for name, label in name_to_label.items():
                    if pred_idx == label and conf >= 0.90:
                        matched_name = name
                        break

                # Merkel/Rutte via transfer
                if not matched_name:
                    embedding = resnet(face_tensor.unsqueeze(0).to(device)).detach().cpu().numpy()[0]
                    if is_match(embedding, rutte_embeddings)[0]:
                        matched_name = "mark rutte"
                    elif is_match(embedding, merkel_embeddings)[0]:
                        matched_name = "angela merkel"

                if matched_name:
                    actor_key = matched_name.replace(" ", "_")
                    landmarks = process_observation_for_actor(image, actor_key)
                    if landmarks:
                        image_landmarks.update(landmarks)

        image_landmarks["image_path"] = image_path
        landmarks_data.append(image_landmarks)

    return pd.DataFrame(landmarks_data)

# run body parts localization
df_results_body_parts = detect_landmarks(images_paths)
df_results_body_parts

In [ ]:
# Join face results with body parts localizations
df_results = pd.merge(df_results_face, df_results_body_parts,  on="image_path", how="left")
